# llminfer Quickstart (Colab)

A clean, fast path for first use:
- setup + install,
- single prompt inference,
- streaming,
- static vs continuous batching,
- small benchmark with plots.



## 0) Runtime
In Colab, choose **Runtime -> Change runtime type -> GPU** before running.


In [ ]:
!nvidia-smi

import os
import platform
import torch

print('Python :', platform.python_version())
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
else:
    print('Warning: CUDA is not enabled. Use a GPU runtime for realistic results.')


## 1) Clone + install

In [ ]:
REPO_URL = 'https://github.com/nickforce989/llminfer.git'
TARGET_DIR = '/content/llminfer'

import os
import shutil

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

!git clone -q {REPO_URL} {TARGET_DIR}
%cd {TARGET_DIR}

!pip -q install -U pip
!pip -q install -e ".[serve,peft]"


## 2) Helpers

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

def header(title: str):
    print('\n' + '=' * 20 + f' {title} ' + '=' * 20)

def show_image(path: str):
    p = Path(path)
    if p.exists():
        print(f'[plot] {p}')
        display(Image(filename=str(p)))
    else:
        print(f'[missing plot] {p}')


## 3) Basic inference

In [ ]:
from llminfer import InferenceEngine, EngineConfig

MODEL = 'facebook/opt-125m'
engine = InferenceEngine(EngineConfig(model_name=MODEL))
engine.load()

header('Single prompt')
result = engine.run(
    'Explain why GPUs are useful for LLM inference in 4 short bullet points.',
    max_new_tokens=120,
    temperature=0.2,
    no_repeat_ngram_size=3,
    bad_words=['joking', 'not sure'],
)
print(result.generated_text)
print('\nStats:', result.stats)


## 4) Streaming output

In [ ]:
header('Streaming demo')
for chunk in engine.stream(
    'Explain KV cache in under 70 words.',
    max_new_tokens=96,
    temperature=0.2,
    no_repeat_ngram_size=3,
):
    if not chunk.is_final:
        print(chunk.token, end='', flush=True)
    else:
        print('\n\nFinal stats:', chunk.stats)


## 5) Static vs continuous batching

In [ ]:
prompts = [
    'Give one line about CUDA graphs.',
    'Give one line about tensor parallelism.',
    'Give one line about paged KV cache.',
    'Give one line about speculative decoding.',
]

header('run_batch (static)')
static_res = engine.run_batch(prompts, max_new_tokens=48, temperature=0.2)
for i, r in enumerate(static_res, 1):
    print(f'{i}. {r.generated_text.strip()}')

header('run_batch_continuous')
cont_res = engine.run_batch_continuous(prompts, max_new_tokens=48, temperature=0.2)
for i, r in enumerate(cont_res, 1):
    print(f'{i}. {r.generated_text.strip()}')


## 6) Paged KV cache (data structure demo)

In [ ]:
import torch
from llminfer.config import CacheConfig
from llminfer.kv_cache import KVCacheManager

cache = KVCacheManager(CacheConfig(enable_paged_kv=True, page_size_tokens=4))
fake_kv = (((torch.zeros(1, 2, 12, 8), torch.zeros(1, 2, 12, 8)),),)[0]
fake_kv = (fake_kv[0],)
cache.update('demo-seq', fake_kv, seq_len=12)
_ = cache.get('demo-seq')
header('Paged KV stats')
print(cache.stats())


## 7) Benchmark with plots

In [ ]:
from llminfer import Benchmarker

bm = Benchmarker(engine)
bench = bm.run(batch_sizes=[1, 2, 4], num_runs=3, warmup_runs=1, max_new_tokens=48, use_continuous_batching=True)
bench.print_summary()

outdir = Path('benchmarks/quickstart')
outdir.mkdir(parents=True, exist_ok=True)
bench.save_json(str(outdir / 'benchmark.json'))
bench.save_csv(str(outdir / 'benchmark.csv'))
plots = bench.plot_suite(str(outdir), prefix='quickstart')
print('Saved artifacts to', outdir.resolve())
print('Plots:', plots)

for name in ['quickstart_dashboard.png', 'quickstart_throughput.png', 'quickstart_latency.png', 'quickstart_memory.png']:
    show_image(str(outdir / name))


## 8) Cleanup

In [ ]:
try:
    engine.unload()
except Exception:
    pass
print('Done.')
